# NVSHMEM (PGAS for GPUs)

NVSHMEM is NVIDIA's implementation of the OpenSHMEM programming model for heterogeneous systems.
It provides a **Partitioned Global Address Space (PGAS)** abstraction that
* allows directly accessing memory on remote GPUs using one-sided operations (get/ put),
* from GPU threads through a device-side API,
* without requiring the remote side to post a matching receive.

The one-sided features that we will be using contrast the MPI concepts we used in [11-mpi](./11-mpi.ipynb) and [13-nvshmem](./13-nvshmem.ipynb), most notably:
* One-sided communication instead of two-sided. MPI also supports a one-sided communication model which we don't cover in this course.
* Device-initiated communication: MPI does not natively provide a device-side API - communication has to be initiated from host code. This can increase latency, limit performance and increase code complexity.
* Automatic overlap: since communication can be initiated by GPU threads, it is straight-forward to fuse and overlap it with computation.

NVSHMEM also features collective operations, similar to MPI, although in NVSHMEM the result of such operations is implicitly broadcasted (at the time of writing).
`nvshmem_double_sum_reduce`, for instance, behaves conceptually akin to an `MPI_Allreduce` or an `MPI_Reduce` with a subsequent `MPI_Broadcast`.

## Initialization

As with MPI, NVSHMEM requires initialization and finalization.

```c++
int main(int argc, char *argv[]) {
    nvshmem_init();

    // ...

    nvshmem_finalize();
}```


## Compilation and Execution

NVSHMEM programs can be compiled with the `nvcc`, but require relocatable device code to be enabled (`-rdc=true`), and linking against additional libraries (`-lnvshmem`).

Execution can be done via a packaged launcher (`nvshmrun`) or directly via `mpirun`.

In many cases, developers want to use MPI and NVSHMEM alongside.
This can be done by initializing NVSHMEM with MPI.

```c++
    // initialize MPI
    MPI_Init(&argc, &argv);

    // initialize NVSHMEM
    nvshmemx_init_attr_t attr;
    MPI_Comm comm = MPI_COMM_WORLD;
    attr.mpi_comm = &comm;
    nvshmemx_init_attr(NVSHMEMX_INIT_WITH_MPI_COMM, &attr);

    // ...

    nvshmem_finalize();

    MPI_Finalize();
```

Compilation can then be done, e.g., via `nvcc -ccbin=mpic++ my-app.cu -rdc=true -lnvshmem` and execution with `mpirun` .

## Thread Control

NVSHMEM uses the concept of **processing elements (PEs)** and **teams**.
If you have an MPI background, the closest match would be **ranks** and **communicators**.

Similar to MPI, multiple PEs are launched which then each execute the same program.
Inside the program, information about them can be queried using API functions.

```c++
int numPEs = nvshmem_n_pes();
int pe     = nvshmem_my_pe();

std::cout << "PE " << pe << " of " << numPEs << std::endl;
```

Note that these API functions also have a device version, i.e. the same information can be queried directly in kernels.

For synchronization between the PEs, `nvshmem_barrier_all()` is available.

Selecting GPUs works analogous to before:

```c++
    int numDevicesPerNode = 0;
    checkCudaError(cudaGetDeviceCount(&numDevicesPerNode));

    int deviceId = pe % numDevicesPerNode;
    checkCudaError(cudaSetDevice(deviceId));
```


## Symmetric Heap

NVSHMEM is based on the idea of **symmetric memory**.
All PEs allocate one or more arrays with identical data type and size.
This allows placing them at identical virtual addresses on each PE.

Allocation is done via
```c++
auto ptr = (double *)nvshmem_malloc(size_per_PE_in_bytes);
```


<div style="text-align: center;">
  <img src="./img/symmetric-heap.png" width="720" style="background-color:white" alt="Symmetric Heap Visualization">
</div>

This mode of organizing allocations allows accessing data on remote PEs.
Since the addresses are symmetric, the exact same pointer is not only valid for the current PE, but also for all others.
We will use the `nvshmem_double_g` function (g is short for get), which takes two parameters:
* a symmetric address, and
* the PE to read from.

Directly using a pointer as local, either via direct access (`ptr[idx]`) or in other API functions such as `cudaMemcpy`, is valid.

Note: Symmetric allocations are done using the **symmetric heap**.
It has a limited size which might need to be adjusted via setting the environment variable `NVSHMEM_SYMMETRIC_SIZE`.

## Stencil Kernel

Our next step is revising the stencil kernel to directly access neighboring PEs.
Note that we keep the original data layout, i.e. including ghost layers, for easier porting.
In practice, these could now be shaved away.

We start by querying PE information for each GPU thread.
This could later be optimized to happen only if necessary and not by default.

```c++
__global__ void stencil2D(...) {
    // ... determine i0 and i1

    // get PE info directly in the kernel
    const auto pe = nvshmem_my_pe();
    const auto numPEs = nvshmem_n_pes();
```


Next, we set up the contributions for left and right neighbors, which we know need to be local due to our 1D decomposition.

```c++
    if (i0 < globalInnerEndX && i1 < globalInnerEndY) {
        // temporary variables for stencil contributions
        double left, right, top, bottom;

        // left and right are always local
        left   = u[(i0 - 1) + i1 * globalNumCellsX];
        right  = u[(i0 + 1) + i1 * globalNumCellsX];
```


For the access to the bottom neighbor, there are three cases:
* We are **not** in the very first (inner) row of the current patch -> this is just a regular access.
* We are in the first row, and there is no patch below ours (`pe == 0`) -> this is an access into the boundary, which is just a regular access
* Otherwise -> this is a remote access

For remote accesses, we need to reconstruct the index **local to the remote PE** and know its PE index.
The bottom neighbor of the current patches **first row** is the **last row** of the patch below.
As before, we need to take ghost layers into account.

```c++
        // top and bottom may be remote
        if (i1 == globalInnerBeginY && pe > 0) {
            // fetch from neighboring PE below
            auto remotePE = pe - 1;
            auto remoteIndex = i0 + (patchHeight + 2 - 2) * globalNumCellsX;
            bottom = nvshmem_double_g(&u[remoteIndex], remotePE);
        } else {
            bottom = u[i0 + (i1 - 1) * globalNumCellsX];
        }
```

Note that this way of indexing works for all but the last patch when the last patch has fewer rows.
The last patch, however, is never targeted as remote PE in this code section.

Access to the top neighbor is handled similarly.
For remote access, and taking ghost layers into account, this then maps to the first **inner row** of the next patch.

```c++
        if (i1 == globalInnerEndY - 1 && pe < numPEs - 1) {
            // fetch from neighboring PE above
            auto remotePE = pe + 1;
            auto remoteIndex = i0 + 1 * globalNumCellsX;
            top = nvshmem_double_g(&u[remoteIndex], remotePE);
        } else {
            top = u[i0 + (i1 + 1) * globalNumCellsX];
        }
```


The last step is computing the final result and writing it back to `uNew`.

```c++
        uNew[i0 + i1 * globalNumCellsX] = u[i0 + i1 * globalNumCellsX] + alpha * (left + right + top + bottom - 4 * u[i0 + i1 * globalNumCellsX]);
    }
}
```

Note that this implements a **pull pattern** - data is fetched on demand.
One alternative is a **push pattern** where new data points are put into the memory of remote PEs as soon as it is produced.

## Reductions

To aggregate the total temperature at the end of our application, we still need to do a reduction across all PEs.
Using an NVSHMEM reduction would look like below, but requires the data to reside in GPU memory.
To facilitate our implementation, we keep the original MPI reduction of host data.

```c++
nvshmem_double_sum_reduce(
    NVSHMEM_TEAM_WORLD,
    &totalTemperature,
    &peTotalTemperature,
    1);
```


## Exercise

Choose one of the difficulty levels below to tailor the exercise to your preferences.
Each level provides a different starting point implementation to be copied into [stencil-2d.cu](../src/13-nvshmem/stencil-2d.cu).
Use it for your implementation and follow the steps outlined above.

Below the difficulty level descriptions, there are cells for compiling, executing and profiling the your solution.

### Level Hard

Create and empty file [stencil-2d.cu](../src/13-nvshmem/stencil-2d.cu) and copy one of the previous code versions into it (your work or a solution).

In [ ]:
!touch ../src/13-nvshmem/stencil-2d.cu

### Level Medium

[stencil-2d-medium.cu](../src/13-nvshmem/stencil-2d-medium.cu) contains a partial solution with TODOs ranging from straight-forward to complex.
Copy the provided code into the working file [stencil-2d.cu](../src/13-nvshmem/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
%%bash
if [ -e ../src/13-nvshmem/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/13-nvshmem/stencil-2d-medium.cu ../src/13-nvshmem/stencil-2d.cu
fi

### Level Easier

[stencil-2d-easier.cu](../src/13-nvshmem/stencil-2d-easier.cu) contains a partial solution with TODOs.
The solution is further progressed than the level medium version, and the complexity of the TODOs is limited.
Copy the provided code into the working file [stencil-2d.cu](../src/13-nvshmem/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
%%bash
if [ -e ../src/13-nvshmem/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/13-nvshmem/stencil-2d-easier.cu ../src/13-nvshmem/stencil-2d.cu
fi

### Possible Solution

[stencil-2d-solution.cu](../src/13-nvshmem/stencil-2d-solution.cu) contains a possible solution for this exercise.
Copy it into the working file [stencil-2d.cu](../src/13-nvshmem/stencil-2d.cu) with the cell below.

In [ ]:
%%bash
if [ -e ../src/13-nvshmem/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/13-nvshmem/stencil-2d-solution.cu ../src/13-nvshmem/stencil-2d.cu
fi

### Compilation, Execution and Profiling

The new code version is available in [13-nvshmem/stencil-2d.cu](../src/13-nvshmem/stencil-2d.cu) (after creating it with one of the commands above).
It can be compiled and executed with the following cells.

In [ ]:
!nvcc -ccbin=mpic++ -O3 -rdc=true -gencode arch=compute_80,code=sm_80 -gencode arch=compute_86,code=sm_86 -o ../build/13-nvshmem ../src/13-nvshmem/stencil-2d.cu -lnvshmem

The next cell produces output for a small grid.
Visualize the output using the [visualize](./99-visualize.ipynb) notebook after executing the application.

By default, it runs 2 ranks on a single GPU.
Use this to verify that your application works as intended.

In [ ]:
!mpirun -n 2 ../build/13-nvshmem 256 64 2 2000 100

The next cell produces no output and runs a larger grid.
Use it for performance evaluation.

In [ ]:
!mpirun -n 2 ../build/13-nvshmem $((32 * 1024)) 256 2 8 0

Instead of running on the local **single A40** GPU, we can also submit a batch job running on **multiple A100** GPUs.
Feel free to tune the number of GPUs (both in the `--gres=gpu:a100:NGPU` and in the `mpirun -n NGPU`) to anything between one and eight GPUs.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/13-nvshmem.out --error=../output/13-nvshmem.err \
    --wrap="mpirun -n 2 ../build/13-nvshmem $((32 * 1024)) 256 2 8 0"

cat ../output/13-nvshmem.out

The next cell performs profiling with Nsight Systems by submitting a batch job.
Feel free to tune the number of GPUs (both in the `--gres=gpu:a100:NGPU` and in the `mpirun -n NGPU`) to anything between one and eight GPUs.

The profile is then available at [profiles/13-nvshmem.nsys-rep](../profiles/13-nvshmem.nsys-rep) and can be downloaded by **shift + right-clicking** the link, by clicking the link with the **middle mouse button**, or using the JupyterHub file tree.

After downloading it, open it up locally to visualize the run-time behavior of your application.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/13-nvshmem-nsys.out --error=../output/13-nvshmem-nsys.err \
    --wrap="nsys profile --stats=true --force-overwrite=true \
        -o ../profiles/13-nvshmem \
        mpirun -n 2 \
            ../build/13-nvshmem $((32 * 1024)) 256 2 8 0"

cat ../output/13-nvshmem-nsys.out

## Next Step

The last step is heading into the future learning section in [14](./14-outlook.ipynb).